### Preprocessing Step 5:

Power curve filtering, fleet median and idiosyncratic components, and min-max scaling

In [1]:
from glob import glob
import pandas as pd
import os
import matplotlib.pyplot as plt
import math
from helperfunctions import intern_constants as ic
from helperfunctions.preprocessing import PreprocessingStep5 as pre
from helperfunctions.preprocessing import PreProcKeys as pkeys

In [2]:
print(pd.__version__)

3.0.2


In [2]:
cfg = {
    pkeys.VERSION           : "v2",
    pkeys.TRAIN_START       : "2018-04-05 13:50:00",
    pkeys.TRAIN_END         : "2019-04-05 13:50:00",
    pkeys.IMPUTED_PATH      : ic.PATH_IMPUTED,
    pkeys.CLEANED_DATA_PATH : ic.PATH_PC_FILTERING, # used for training
    pkeys.MINMAX_FILENAME   : ic.PATH_MINMAX_SCALER,
    pkeys.PC_MASKS_PATH     : ic.PATH_POWERCURVE_MASKS,
    pkeys.IDIO_PATH         : ic.PATH_IDIO_COMP,
    pkeys.FM_PATH           : ic.PATH_FLEETMEDIAN,
    pkeys.EXCLUDECOLLIST    : [ic.WT_ID, ic.TS_COL],
}

In [4]:
pre().execute_pre_step5(cfg)

creating pc masks:   0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:45<00:00,  3.27s/it]


TypeError: DataFrame.groupby() got an unexpected keyword argument 'axis'

### Optional Plotting
Visualization of raw signals, idiosyncratic components, and fleet median

In [ ]:
imp_files   = sorted(glob(os.path.join(cfg["imputed_path"], "*.csv")))
idio_files  = sorted(glob(os.path.join(cfg["idio_path"], "*.csv")))
fm_files    = sorted(glob(os.path.join(cfg["fm_path"], "*.csv")))

df_imp = pd.read_csv(imp_files[0], parse_dates= [ic.TS_COL], index_col=ic.TS_COL)
all_signals = [col for col in df_imp.columns if col != ic.WT_ID]

n_cols = 5
n_rows = math.ceil(len(all_signals) / n_cols)
figscale = 4

df_fm = pd.read_csv(fm_files[0], parse_dates= [ic.TS_COL], index_col=ic.TS_COL)

for imp, idio in list(zip(imp_files, idio_files)):
    df_imp = pd.read_csv(imp, parse_dates= [ic.TS_COL], index_col=ic.TS_COL)
    df_idio = pd.read_csv(idio, parse_dates= [ic.TS_COL], index_col=ic.TS_COL)
    
    
    wt_id = df_imp[ic.WT_ID].iat[0]
    if (df_idio[ic.WT_ID].iat[0] == wt_id):
        fig, axs = plt.subplots(n_rows, n_cols,
                               figsize=(figscale * n_cols, figscale* n_rows),
                               sharex=False, sharey = False)
        # loc = mdates.AutoDateLocator()
        # fmt = mdates.ConciseDateFormatter(loc)
        if n_rows ==1:
            axs = axs.reshape(1,-1)
        
        for i, sig in enumerate(all_signals):
            r, c = divmod(i, n_cols)
            ax = axs[r,c]
            ax.plot(df_imp.index, df_imp[sig], label="original", linewidth=0.5)
            ax.plot(df_idio.index, df_idio[sig], label="idios comp.", linewidth=0.5)
            ax.plot(df_fm.index, df_fm[sig], label="fleet median", linewidth=0.5)
            ax.set_title(sig, fontsize=11)
            ax.legend(fontsize=10, loc="lower right", frameon=False)
            ax.grid(True, alpha=0.3)
           
        # for ax in axs.flat:
        #     ax.xaxis.set_major_locator(loc)
        #     ax.xaxis.set_major_formatter(fmt)
        #     ax.tick_params(axis='x', rotation=30)
            
        for j in range(len(all_signals), n_rows * n_cols):
            fig.delaxes(axs.flat[j])
        #fig.autofmt_xdate()     
        fig.suptitle(f"WT_ID {wt_id}", fontsize=12)
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        plt.show()
    